# 🎯 [ISR02_M08] 선전전 폭증 모의 대량 SNS 스트림 데이터 유입 부하 시뮬레이션 테스트
### Google Colab 실습 환경 (정보·감시·정찰(ISR) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR02/isr02_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 2) |
| **미션 ID** | ISR02_M08 |
| **미션 주제** | 선전전 폭증 모의 대량 SNS 스트림 데이터 유입 부하 시뮬레이션 테스트 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 텔레그램·X 같은 SNS에 현장 제보와 함께 재게시, 수치·지역 변조, 과거 사진 재활용, 생성형 AI 조작물, 조직적 동시 유포가 한꺼번에 쏟아지면서 공개출처정보(OSINT)의 출처와 신뢰도를 빠르게 가려내는 일이 정보전의 핵심이 되었다. 본 프로젝트에서는 가상 계정과 가상 지역 코드로 만든 교육용 합성 게시물을 사용해 원천 데이터 스키마 정의, 텍스트 정제와 이미지 지각 해시 기반 중복 제거, 유사도 기반 복제·1차 출처 분류 Baseline, 조작·기만 탐지와 계정 신뢰도 결합 모델, 종합 신뢰도 등급(A~E) 엔진, 정보전 모니터링 대시보드, 정보관 긴급 검토(HITL), 대량 스트림 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 조작 게시물을 놓치지 않으면서 사실 제보를 가짜로 몰지 않고, 파급력이 큰 첩보의 채택·차단은 원천 출처를 확인한 정보관이 결정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/isr02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/` | 전처리, Baseline, 앙상블 탐지, 대시보드, HITL 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/isr02_data_spec.md`, `docs/isr02_hitl_guide.md`, `docs/isr02_aar_report.md`, `docs/isr02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 군사 행정/지휘관 문서 |
| **`configs/`**| `configs/isr02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/isr02_stress_test.py` | 500% 부하 시험 및 단위/통합 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/isr02_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 선전전 폭증 모의 대량 SNS 스트림 데이터 유입 부하 시뮬레이션 테스트
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

선전 공작이 집중되는 상황을 모의하여 초당 수천 건의 SNS 게시물 유입 부하 스트레스 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **선전 공작이 집중되는 상황을 모의해 초당 수천 건의 SNS 게시물 유입 부하 스트레스 테스트를 수행한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **평시 초당 1,000건과 500% 증가한 초당 6,000건 스트림, 손상 게시물 혼입, 처리 자원 축소 상황을 가상으로 주입하여 처리 지연 및 복원력을 시험하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `tests/isr02_stress_test.py 및 스트레스 테스트 분석 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 극단 상황에서의 안전 실패(Fail-safe) 및 기능 축소 작동 적절성

#### 미션 PASS 조건

- 정량 기준: 동시 트래픽 500% 증가 시 시스템 다운 0건, 데이터 유실률 0%


- 진행 기준: 대량 시뮬레이션 부하 주입 테스트 통과 및 병목 구간 프로파일링 완료


- 검증 방법: 부하 테스트 도구를 통한 처리량(Throughput) 및 메모리 누수 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""선전 공작이 집중되는 상황을 모의해 초당 1,000건(평시)에서 6,000건(500% 증가)까지 SNS 게시물 스트림을 주입하고, 마이크로 배치 처리·유한 대기열·초과분 보관 버퍼·기능 축소(이미지 분석 지연) 모드로 시스템 다운 0건·데이터 유실 0%를 지키는지, 처리량·지연·메모리·병목을 검증한다."""
import re
import time
import tracemalloc
import unicodedata
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.fft import dctn

# ---------------------------------------------------------------------------
# [교육용가상자료] 합성 SNS 게시물 생성기 (실제 계정·채널·지명·게시물 없음)
# 가상 사건마다 원본 제보, 재게시(복제), 유사 재작성, 조작·기만 게시물 10종과 32x32 합성 이미지를 만든다.
# ---------------------------------------------------------------------------
REGIONS = [f"{a}-{n}" for a in "ABCDEF" for n in range(1, 6)]
OBJECTS = ["군용 차량", "소형 드론", "보급 트럭", "헬기", "버스", "견인 차량", "트레일러", "민간 승용차"]
DIRS = ["북쪽", "남쪽", "동쪽", "서쪽"]
TEMPLATES = ["가상 {r} 지역에서 {o} 목격, {n}대 규모로 {d}으로 이동 중",
             "{r} 일대 {o} 확인됨. 약 {n}대, {d} 방향으로 이동",
             "[현지] {r}에서 {o} 발견 - 수량 {n}, 이동 방향 {d}",
             "{r} 주민 제보: {o} {n}대가 {d}으로 향하는 모습"]
MANIP = ["num_change", "region_swap", "dir_flip", "old_image", "splice", "ai_image", "ai_text", "bot_burst", "mashup", "recycled"]
ACCOUNT_TYPES = {  # 유형별 계정 속성 분포(가상): 나이(일), 팔로워, 하루 게시 수, 원본 비율, 과거 정확도, 게시 간격 변동계수
    "media": ((1000, 4000), (5e4, 5e5), (5, 30), (.6, .9), (.85, .97), (.8, 1.5)),
    "witness": ((10, 800), (20, 2000), (.2, 3), (.7, 1.0), (.6, .95), (1.0, 2.0)),
    "aggregator": ((200, 2000), (1e4, 2e5), (30, 200), (.05, .2), (.6, .85), (.5, 1.0)),
    "propaganda": ((5, 200), (100, 2e4), (20, 150), (.3, .7), (.1, .4), (.05, .3)),
}


def make_accounts(rng, n=(8, 30, 10, 20)):
    acc = {}
    for (t, spec), k in zip(ACCOUNT_TYPES.items(), n):
        for i in range(k):
            v = [rng.uniform(*r) for r in spec]
            acc[f"ACC_{t[:3].upper()}{i:02d}"] = {"type": t, "age_days": int(v[0]), "followers": int(v[1]), "posts_per_day": round(v[2], 1),
                                                   "original_ratio": round(v[3], 2), "past_accuracy": round(v[4], 2), "interval_cv": round(v[5], 2)}
    return acc


def event_image(seed):
    """사건 현장 사진을 흉내 낸 32x32 합성 이미지(사건마다 고유한 도형 배치)."""
    r = np.random.default_rng(seed)
    img = np.linspace(r.uniform(60, 120), r.uniform(120, 200), 32)[:, None] * np.ones((1, 32))
    for _ in range(int(r.integers(3, 7))):
        y, x, h, w = r.integers(0, 26, 4)
        img[y:y + 3 + h % 8, x:x + 3 + w % 8] = r.uniform(20, 240)
    return img + r.normal(0, 4, (32, 32))


def variant(img, rng):
    """재게시 과정의 자연스러운 변형: 재압축 잡음, 밝기 변화, 가장자리 잘림 중 하나."""
    k = rng.integers(0, 3)
    if k == 0:
        return ndimage.gaussian_filter(img, 0.6) + rng.normal(0, 3, img.shape)
    if k == 1:
        return img * rng.uniform(0.85, 1.15)
    return ndimage.zoom(img[2:30, 2:30], 32 / 28, order=1)


def splice(img, other, rng):
    out = img.copy()
    y, x = rng.integers(4, 18, 2)
    out[y:y + 12, x:x + 12] = other[y:y + 12, x:x + 12] + rng.normal(0, 12, (12, 12))   # 잡음 수준이 다른 조각을 붙임
    return out


def ai_image(rng):
    """생성형 모델 이미지를 흉내 낸 합성 이미지: 지나치게 매끈한 바탕 + 업샘플링 격자 무늬(주기 2픽셀)."""
    base = ndimage.gaussian_filter(rng.normal(130, 60, (32, 32)), 3)
    grid = (np.indices((32, 32)).sum(0) % 2) * rng.uniform(6, 12)
    return base + grid


def write(ev, rng, tpl=None, **kw):
    s = {"r": ev["r"], "o": ev["o"], "n": ev["n"], "d": ev["d"], **kw}
    return (tpl or TEMPLATES[int(rng.integers(len(TEMPLATES)))]).format(**s)


def make_posts(n_events=40, seed=7):
    rng = np.random.default_rng(seed)
    acc = make_accounts(rng)
    by = {t: [a for a, v in acc.items() if v["type"] == t] for t in ACCOUNT_TYPES}
    events = [{"id": f"EV{i:03d}", "r": str(rng.choice(REGIONS)), "o": str(rng.choice(OBJECTS)), "n": int(rng.integers(2, 15)),
               "d": str(rng.choice(DIRS)), "t0": float(i * 1.2 + rng.uniform(0, 1)), "img": event_image(seed * 1000 + i)} for i in range(n_events)]
    posts = []

    def add(ev, account, text, img, t, fwd, kind, origin):
        pid = f"P{len(posts):04d}"
        posts.append({"post_id": pid, "platform": str(rng.choice(["telegram", "x", "other"], p=[.6, .3, .1])), "account_id": account,
                      "posted_at": round(t, 3), "text": text, "image": img, "forwarded_from": fwd, "event_id": ev["id"] if ev else None,
                      "kind": kind, "is_disinfo": kind in MANIP, "origin": origin})
        return pid

    for k, ev in enumerate(events):
        src = str(rng.choice(by["witness"] if rng.random() < .6 else by["media"]))
        if rng.random() < .05:
            src = str(rng.choice(by["propaganda"]))                      # 신뢰도 낮은 계정도 사실을 올릴 수 있다
        o = add(ev, src, write(ev, rng), ev["img"], ev["t0"], None, "original", None)
        for _ in range(int(rng.integers(2, 5))):                          # 그대로 재게시(전달)
            add(ev, str(rng.choice(by["aggregator"])), posts[int(o[1:])]["text"], ev["img"], ev["t0"] + rng.exponential(1), o, "copy", o)
        for _ in range(int(rng.integers(1, 3))):                          # 표현만 바꾼 재작성(내용 동일)
            add(ev, str(rng.choice(by["media"] + by["witness"])), "#가상속보 " + write(ev, rng), variant(ev["img"], rng),
                ev["t0"] + rng.uniform(0.5, 6), None, "near_dup", o)
        for m in MANIP if k % 2 == 0 else MANIP[k % 4::4]:                # 조작·기만 게시물
            if rng.random() > 0.5:
                continue
            who = str(rng.choice(by["propaganda"] if rng.random() < .75 else by["aggregator"] + by["witness"]))
            t, img, text = ev["t0"] + rng.uniform(1, 12), variant(ev["img"], rng), None
            if m == "num_change":
                text = write(ev, rng, n=ev["n"] * int(rng.integers(5, 10)))
            elif m == "region_swap":
                text = write(ev, rng, r=str(rng.choice([r for r in REGIONS if r != ev["r"]])))
            elif m == "dir_flip":
                text = write(ev, rng, d={"북쪽": "남쪽", "남쪽": "북쪽", "동쪽": "서쪽", "서쪽": "동쪽"}[ev["d"]])
            elif m == "old_image" and k > 0:
                text, img = write(ev, rng), events[int(rng.integers(0, k))]["img"]
            elif m == "splice":
                text, img = write(ev, rng), splice(ev["img"], events[(k + 7) % n_events]["img"], rng)
            elif m == "ai_image":
                text, img = write(ev, rng), ai_image(rng)
            elif m == "ai_text":
                text = f"속보 속보! {ev['r']} 지역 대규모 {ev['o']} 출현! 긴급 확산 바람! 긴급 확산 바람! 모두 확산 바람!"
                img = ai_image(rng) if rng.random() < .5 else None
            elif m == "bot_burst":
                text = write(ev, rng, n=ev["n"] * 6) + " 지금 바로 공유"
                for b in rng.choice(by["propaganda"], 4, replace=False):
                    add(ev, str(b), text, img, t + rng.uniform(0, 0.15), None, m, o)
                continue
            elif m == "mashup":
                other = events[(k + 3) % n_events]
                text = f"{ev['r']} 일대 {ev['o']} 확인, {other['n']}대가 {other['d']}으로 이동"   # 다른 사건 내용과 짜깁기
            elif m == "recycled" and k > 2:
                old = events[int(rng.integers(0, k - 1))]
                text, img, t = write(old, rng), old["img"], ev["t0"] + rng.uniform(0, 3)
            if text is None:
                continue
            add(ev, who, text, img, t, None, m, o)
    return posts, acc
# ---------------------------------------------------------------------------

# ---- 텍스트 정제·이미지 지각 해시·조작 흔적 공통 함수 ----
STOPWORDS = {"가상", "약", "일대", "지역에서", "모습", "중", "확인됨", "발견", "제보", "주민"}
_URL = re.compile(r"https?://\S+|www\.\S+")
_SPECIAL = re.compile(r"[^0-9A-Za-z가-힣\-\s]")


def clean_text(t):
    """정규화(NFKC) -> URL·멘션 제거 -> 해시태그 기호 제거 -> 특수문자 제거 -> 불용어 제거 -> 공백 정리."""
    t = unicodedata.normalize("NFKC", t)
    t = _URL.sub(" ", t)
    t = re.sub(r"@\w+", " ", t).replace("#", " ")
    t = _SPECIAL.sub(" ", t)
    return " ".join(w for w in t.split() if w not in STOPWORDS)


def phash(img):
    """지각 해시(pHash): 32x32 이미지 DCT의 저주파 8x8(직류 성분 제외)을 중앙값과 비교한 64비트."""
    c = dctn(np.asarray(img, float), norm="ortho")[:8, :8].ravel()[1:]
    return np.append(c > np.median(c), False)


def hamming(a, b):
    return int(np.count_nonzero(a != b))


def slots(text):
    """본문에서 지역 코드·수량·방향·대상을 뽑는다(사건 내용 비교용)."""
    r = re.search(r"[A-F]-\d", text)
    n = re.search(r"(\d+)\s*대|수량\s*(\d+)", text)
    d = next((x for x in DIRS if x in text), None)
    o = next((x for x in OBJECTS if x in text), None)
    return {"r": r.group(0) if r else None, "n": int(n.group(1) or n.group(2)) if n else None, "d": d, "o": o}


def splice_score(img):
    """합성(붙여넣기) 흔적: 8x8 블록별 고주파 잡음 크기의 최대/중앙값 비율. 잡음 수준이 다른 조각이 있으면 커진다."""
    res = img - ndimage.median_filter(img, 3)
    b = np.abs(res).reshape(4, 8, 4, 8).std(axis=(1, 3))
    return float(b.max() / (np.median(b) + 1e-6))


def ai_image_score(img):
    """생성형 이미지 흔적: 2픽셀 주기 격자(나이퀴스트 주파수) 에너지 비율."""
    f = np.abs(np.fft.fft2(img - img.mean()))
    return float(f[16, 16] / (f.sum() / f.size + 1e-6))


def ai_text_score(text):
    """기계적 선동 문체 흔적: 반복 어절 비율 + 느낌표 밀도."""
    w = text.split()
    return (1 - len(set(w)) / max(len(w), 1)) + text.count("!") / max(len(w), 1)

QUEUE_CAP = 8000          # 메모리 대기열 용량(건). 넘치면 보관 버퍼로 옮겨 유실을 막는다
pool, _ = make_posts(60, seed=7)


def stream(n, broken=0.0, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n):
        p = dict(pool[i % len(pool)]); p["post_id"] = f"S{i:06d}"
        r = rng.random()
        if r < broken / 3:
            p["text"] = None
        elif r < 2 * broken / 3:
            p["image"] = np.zeros((5, 5))
        elif r < broken:
            p["text"] = 12345
        out.append(p)
    return out


def process(batch, full, prof, seen):
    """마이크로 배치 처리: 본문 정제 -> 중복 해시 -> (정상 모드만) 이미지 해시 -> 선동 문체 점수. 형식 오류는 격리."""
    done = deferred = quarantined = 0
    for p in batch:
        try:
            if not isinstance(p["text"], (str, type(None))):
                raise TypeError("본문 형식 오류")
            t0 = time.perf_counter(); txt = clean_text(p["text"] or ""); prof["정제"] += time.perf_counter() - t0
            seen[hash(txt)] = seen.get(hash(txt), 0) + 1
            if p["image"] is not None:
                if p["image"].shape != (32, 32):
                    raise ValueError("이미지 형식 오류")
                if full:
                    t0 = time.perf_counter(); phash(p["image"]); prof["이미지 해시"] += time.perf_counter() - t0
                else:
                    deferred += 1                      # 이미지 분석은 나중에 처리(유실 아님)
            t0 = time.perf_counter(); ai_text_score(txt); prof["문체 점수"] += time.perf_counter() - t0
            done += 1
        except Exception:
            quarantined += 1
    return done, deferred, quarantined


# 1) 처리 용량 측정(정상 모드 / 기능 축소 모드)
prof, seen = {"정제": 0.0, "이미지 해시": 0.0, "문체 점수": 0.0}, {}
sample = stream(3000, seed=1)
t0 = time.perf_counter(); process(sample, True, prof, seen); cap_full = len(sample) / (time.perf_counter() - t0)
t0 = time.perf_counter(); process(sample, False, dict.fromkeys(prof, 0.0), {}); cap_deg = len(sample) / (time.perf_counter() - t0)
print(f"처리 용량: 정상 모드 {cap_full:,.0f}건/초, 기능 축소 모드(이미지 분석 지연) {cap_deg:,.0f}건/초")

# 2) 상황별 스트림 주입(3초 분량): 초 단위로 도착분을 대기열에 넣고, 밀리면 기능 축소 모드로 전환
cases = [("평시 1,000건/초", 1000, 0.0, 1.0), ("500% 증가 6,000건/초", 6000, 0.0, 1.0),
         ("6,000건/초 + 손상 게시물 3%", 6000, 0.03, 1.0), ("6,000건/초 + 처리 자원 1/4", 6000, 0.0, 0.25)]
rows = []
for k, (name, rate, br, cpu) in enumerate(cases):
    arrivals = stream(rate * 3, br, seed=10 + k)
    q, spill = deque(), deque()
    done = deferred = quar = down = 0
    max_backlog, modes, lat = 0, [], []
    try:
        for sec in range(6):                               # 3초 도착 + 3초 잔여 처리
            for p in arrivals[sec * rate:(sec + 1) * rate]:
                (q if len(q) < QUEUE_CAP else spill).append(p)
            while len(q) < QUEUE_CAP and spill:
                q.append(spill.popleft())
            backlog = len(q) + len(spill)
            full = backlog <= cap_full * cpu                  # 1초 안에 못 비우면 기능 축소
            budget = int((cap_full if full else cap_deg) * cpu)
            batch = [q.popleft() for _ in range(min(budget, len(q)))]
            d, df_, qu = process(batch, full, dict.fromkeys(prof, 0.0), {})
            done += d; deferred += df_; quar += qu
            modes.append("정상" if full else "축소")
            max_backlog = max(max_backlog, backlog)
            lat.append(backlog / max(budget, 1))
    except Exception:
        down = 1
    left = len(q) + len(spill)
    rows.append({"상황": name, "수신": len(arrivals), "처리": done, "격리": quar, "대기 잔여": left, "유실": len(arrivals) - done - quar - left,
                 "다운": down, "최대 대기(건)": max_backlog, "최대 지연(초)": round(max(lat), 2), "이미지 지연처리": deferred,
                 "모드": "/".join(m[0] for m in modes)})
df = pd.DataFrame(rows)
print(df.drop(columns=["모드"]).to_string(index=False))
print("초별 모드(정=정상, 축=기능 축소):", {r["상황"]: r["모드"] for r in rows})
print(f"\n시스템 다운 {df['다운'].sum()}건, 데이터 유실률 {df['유실'].sum() / df['수신'].sum():.1%} (목표 0%), 대기 잔여분은 보관 버퍼에서 이어서 처리")

# 3) 메모리 안정성과 병목
tracemalloc.start(); mem = []
for _ in range(5):
    process(sample[:1000], True, dict.fromkeys(prof, 0.0), {}); mem.append(tracemalloc.get_traced_memory()[0] / 1e6)
tracemalloc.stop()
print("메모리(동일 배치 5회, MB):", [round(m, 2) for m in mem], f"-> 증가 {mem[-1] - mem[0]:.2f} MB (누수 판단 기준 5MB)")
tot = sum(prof.values())
print("병목 구간(누적 처리시간 비율):", {k: f"{v / tot:.0%}" for k, v in sorted(prof.items(), key=lambda x: -x[1])})
print("개선: 이미지 해시를 별도 비동기 작업자로 분리하면 기능 축소 없이 6,000건/초를 감당할 수 있다(코어 추가 시).")

plt.figure(figsize=(8, 4))
plt.bar(range(len(df)), df["최대 대기(건)"], color=["tab:red" if "축" in m else "tab:blue" for m in df["모드"]])
plt.xticks(range(len(df)), [f"case{i + 1}" for i in range(len(df))]); plt.ylabel("max backlog (posts)")
plt.title("SNS stream stress test (red = degraded mode used)"); plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
처리 용량: 정상 모드 8,868건/초, 기능 축소 모드(이미지 분석 지연) 69,099건/초
                  상황    수신    처리  격리  대기 잔여  유실  다운  최대 대기(건)  최대 지연(초)  이미지 지연처리
         평시 1,000건/초  3000  3000   0      0   0   0      1000      0.11         0
    500% 증가 6,000건/초 18000 18000   0      0   0   0      6000      0.68         0
6,000건/초 + 손상 게시물 3% 18000 17655 345      0   0   0      6000      0.68         0
6,000건/초 + 처리 자원 1/4 18000 18000   0      0   0   0      6000      0.35     17748
초별 모드(정=정상, 축=기능 축소): {'평시 1,000건/초': '정/정/정/정/정/정', '500% 증가 6,000건/초': '정/정/정/정/정/정', '6,000건/초 + 손상 게시물 3%': '정/정/정/정/정/정', '6,000건/초 + 처리 자원 1/4': '축/축/축/정/정/정'}

시스템 다운 0건, 데이터 유실률 0.0% (목표 0%), 대기 잔여분은 보관 버퍼에서 이어서 처리
메모리(동일 배치 5회, MB): [0.01, 0.01, 0.01, 0.01, 0.01] -> 증가 0.00 MB (누수 판단 기준 5MB)
병목 구간(누적 처리시간 비율): {'이미지 해시': '85%', '정제': '11%', '문체 점수': '4%'}
개선: 이미지 해시를 별도 비동기 작업자로 분리하면 기능 축소 없이 6,000건/초를 감당할 수 있다(코어 추가 시).
```
</details>
